# Chapter 11.2: Testing Framework

This notebook covers the testing infrastructure in vLLM and SGLang, including unit tests,
integration tests, and end-to-end tests.

**Learning Objectives:**
- Understand test organization in vLLM and SGLang
- Write unit tests for core components
- Write model correctness tests
- Write API integration tests
- Use fixtures and mocking for GPU-dependent code

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part11/chapter_11.2_testing.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part11/chapter_11.2_testing.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Test Organization Overview

Both vLLM and SGLang use **pytest** as their test framework. Let's understand how tests are organized.

In [ ]:
# Demo: Explore the test organization in vLLM

vllm_test_structure = {
    "tests/": {
        "conftest.py": "Global fixtures (model fixtures, server fixtures)",
        "core/": {
            "test_scheduler.py": "Scheduler logic tests",
            "test_block_manager.py": "Block manager allocation tests",
            "test_block_manager_v2.py": "V2 block manager tests",
        },
        "engine/": {
            "test_async_llm_engine.py": "Async engine tests",
            "test_llm_engine.py": "Sync engine tests",
        },
        "models/": {
            "test_models.py": "Model correctness (compare with HF)",
            "test_big_models.py": "Large model tests (need more GPU)",
        },
        "kernels/": {
            "test_attention.py": "Attention kernel correctness",
            "test_cache.py": "Cache kernel tests",
            "test_pos_encoding.py": "Positional encoding tests",
        },
        "entrypoints/": {
            "test_openai_server.py": "OpenAI API compatibility tests",
            "test_chat.py": "Chat endpoint tests",
        },
        "lora/": {
            "test_lora.py": "LoRA adapter tests",
        },
        "spec_decode/": {
            "test_spec_decode.py": "Speculative decoding tests",
        },
        "distributed/": {
            "test_tensor_parallel.py": "Tensor parallel tests",
        },
    }
}

def print_test_tree(structure, indent=0):
    for name, content in structure.items():
        prefix = "  " * indent
        if isinstance(content, dict):
            print(f"{prefix}{name}")
            print_test_tree(content, indent + 1)
        else:
            print(f"{prefix}{name:40s}  # {content}")

print("vLLM Test Structure")
print("=" * 70)
print_test_tree(vllm_test_structure)

# Test categories and markers
print("\n\nTest Markers Used in vLLM:")
print("-" * 50)
markers = [
    ("@pytest.mark.skip", "Skip test unconditionally"),
    ("@pytest.mark.skipif(not torch.cuda.is_available())", "Skip if no GPU"),
    ("@pytest.mark.parametrize", "Run with multiple parameter sets"),
    ("@pytest.mark.distributed", "Requires multi-GPU"),
    ("@pytest.mark.slow", "Long-running tests"),
]
for marker, desc in markers:
    print(f"  {marker}")
    print(f"    -> {desc}")

In [ ]:
# Demo: pytest configuration used in vLLM

pytest_ini = '''
# pyproject.toml or pytest.ini configuration
[tool.pytest.ini_options]
markers = [
    "slow: marks tests as slow (deselect with '-m not slow')",
    "distributed: marks tests requiring multiple GPUs",
    "gpu: marks tests requiring GPU",
]
testpaths = ["tests"]
timeout = 600
addopts = "-v --tb=short"
filterwarnings = [
    "ignore::DeprecationWarning",
    "ignore::UserWarning",
]
'''

print("pytest configuration:")
print(pytest_ini)

# Common pytest invocations for vLLM development
print("\nCommon pytest commands for vLLM:")
print("=" * 60)
commands = [
    ("pytest tests/core/ -v", "Run all core tests"),
    ("pytest tests/core/test_scheduler.py -v", "Run specific test file"),
    ("pytest tests/core/test_scheduler.py::test_scheduler_add -v", "Run single test"),
    ("pytest tests/ -m 'not slow' -v", "Skip slow tests"),
    ("pytest tests/ -x --timeout=120", "Stop at first failure, 2min timeout"),
    ("pytest tests/ -k 'scheduler and not distributed'", "Filter by name"),
    ("pytest tests/ --co -q", "List tests without running them"),
    ("pytest tests/ -n auto", "Run tests in parallel (pytest-xdist)"),
]
for cmd, desc in commands:
    print(f"  {cmd}")
    print(f"    -> {desc}")

## 2. Unit Tests: Testing Core Components

In [ ]:
# Demo: Writing a unit test for a scheduler component
# This simulates the vLLM scheduler's request management

from dataclasses import dataclass, field
from typing import List, Optional, Dict
from enum import Enum
import time

class SequenceStatus(Enum):
    WAITING = "waiting"
    RUNNING = "running"
    SWAPPED = "swapped"
    FINISHED = "finished"

@dataclass
class SequenceGroup:
    """Simplified version of vLLM's SequenceGroup."""
    request_id: str
    prompt_token_ids: List[int]
    status: SequenceStatus = SequenceStatus.WAITING
    output_token_ids: List[int] = field(default_factory=list)
    arrival_time: float = field(default_factory=time.time)
    
    @property
    def num_tokens(self):
        return len(self.prompt_token_ids) + len(self.output_token_ids)

class SimpleScheduler:
    """Simplified scheduler for testing purposes."""
    
    def __init__(self, max_num_seqs: int = 4, max_num_tokens: int = 1024):
        self.max_num_seqs = max_num_seqs
        self.max_num_tokens = max_num_tokens
        self.waiting: List[SequenceGroup] = []
        self.running: List[SequenceGroup] = []
        self.swapped: List[SequenceGroup] = []
    
    def add_request(self, seq_group: SequenceGroup) -> bool:
        """Add a new request to the waiting queue."""
        if seq_group.status != SequenceStatus.WAITING:
            raise ValueError(f"Expected WAITING status, got {seq_group.status}")
        self.waiting.append(seq_group)
        return True
    
    def schedule(self) -> Dict:
        """Schedule sequences for the next step."""
        scheduled = []
        preempted = []
        total_tokens = sum(sg.num_tokens for sg in self.running)
        
        # First, try to continue running sequences
        new_running = []
        for sg in self.running:
            if sg.status == SequenceStatus.FINISHED:
                continue
            new_running.append(sg)
        self.running = new_running
        
        # Then, admit new sequences from waiting
        while self.waiting and len(self.running) < self.max_num_seqs:
            sg = self.waiting[0]
            new_total = total_tokens + sg.num_tokens
            if new_total > self.max_num_tokens:
                break
            self.waiting.pop(0)
            sg.status = SequenceStatus.RUNNING
            self.running.append(sg)
            scheduled.append(sg)
            total_tokens = new_total
        
        return {
            "scheduled": scheduled,
            "preempted": preempted,
            "num_running": len(self.running),
            "num_waiting": len(self.waiting),
        }
    
    def finish_request(self, request_id: str):
        """Mark a request as finished."""
        for sg in self.running:
            if sg.request_id == request_id:
                sg.status = SequenceStatus.FINISHED
                return True
        return False

print("SimpleScheduler class defined for testing.")
print("This mimics the core logic of vLLM's scheduler.")

In [ ]:
# Demo: Write unit tests for the SimpleScheduler
import unittest

class TestSimpleScheduler(unittest.TestCase):
    """Unit tests for the SimpleScheduler."""
    
    def setUp(self):
        """Set up test fixtures."""
        self.scheduler = SimpleScheduler(max_num_seqs=3, max_num_tokens=100)
    
    def _create_seq_group(self, request_id, num_tokens=10):
        """Helper to create a sequence group."""
        return SequenceGroup(
            request_id=request_id,
            prompt_token_ids=list(range(num_tokens))
        )
    
    def test_add_request(self):
        """Test adding requests to waiting queue."""
        sg = self._create_seq_group("req-1")
        self.scheduler.add_request(sg)
        self.assertEqual(len(self.scheduler.waiting), 1)
        self.assertEqual(self.scheduler.waiting[0].request_id, "req-1")
    
    def test_add_request_invalid_status(self):
        """Test that adding a non-WAITING request raises error."""
        sg = self._create_seq_group("req-1")
        sg.status = SequenceStatus.RUNNING
        with self.assertRaises(ValueError):
            self.scheduler.add_request(sg)
    
    def test_schedule_single_request(self):
        """Test scheduling a single request."""
        sg = self._create_seq_group("req-1", num_tokens=10)
        self.scheduler.add_request(sg)
        result = self.scheduler.schedule()
        
        self.assertEqual(len(result["scheduled"]), 1)
        self.assertEqual(result["num_running"], 1)
        self.assertEqual(result["num_waiting"], 0)
        self.assertEqual(sg.status, SequenceStatus.RUNNING)
    
    def test_schedule_max_seqs(self):
        """Test that scheduler respects max_num_seqs limit."""
        for i in range(5):  # Add 5 requests, but max is 3
            sg = self._create_seq_group(f"req-{i}", num_tokens=10)
            self.scheduler.add_request(sg)
        
        result = self.scheduler.schedule()
        self.assertEqual(result["num_running"], 3)  # Max 3
        self.assertEqual(result["num_waiting"], 2)  # 2 still waiting
    
    def test_schedule_max_tokens(self):
        """Test that scheduler respects max_num_tokens limit."""
        # max_num_tokens = 100, each request has 40 tokens
        for i in range(3):
            sg = self._create_seq_group(f"req-{i}", num_tokens=40)
            self.scheduler.add_request(sg)
        
        result = self.scheduler.schedule()
        # Only 2 can fit (40+40=80 < 100, but 40+40+40=120 > 100)
        self.assertEqual(result["num_running"], 2)
        self.assertEqual(result["num_waiting"], 1)
    
    def test_finish_request(self):
        """Test finishing a request frees a slot."""
        for i in range(3):
            sg = self._create_seq_group(f"req-{i}", num_tokens=10)
            self.scheduler.add_request(sg)
        
        self.scheduler.schedule()  # All 3 running
        self.assertEqual(len(self.scheduler.running), 3)
        
        # Finish one
        self.scheduler.finish_request("req-0")
        
        # Add another
        sg = self._create_seq_group("req-3", num_tokens=10)
        self.scheduler.add_request(sg)
        
        result = self.scheduler.schedule()
        # Should have scheduled the new one
        self.assertEqual(result["num_running"], 3)  # 2 old + 1 new
    
    def test_fifo_ordering(self):
        """Test that requests are scheduled in FIFO order."""
        for i in range(5):
            sg = self._create_seq_group(f"req-{i}", num_tokens=10)
            self.scheduler.add_request(sg)
        
        result = self.scheduler.schedule()
        scheduled_ids = [sg.request_id for sg in result["scheduled"]]
        self.assertEqual(scheduled_ids, ["req-0", "req-1", "req-2"])

# Run the tests
print("Running SimpleScheduler Unit Tests")
print("=" * 60)
suite = unittest.TestLoader().loadTestsFromTestCase(TestSimpleScheduler)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

## 3. Model Correctness Tests

In [ ]:
# Demo: Understanding model correctness tests in vLLM
# These tests compare vLLM outputs with HuggingFace reference outputs

model_test_template = '''
"""Model correctness test - compares vLLM output with HuggingFace.

This is the pattern used in vLLM's tests/models/ directory.
"""
import pytest
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_NAME = "facebook/opt-125m"
MAX_TOKENS = 32
PROMPTS = [
    "Hello, my name is",
    "The capital of France is",
    "The meaning of life is",
]

@pytest.fixture(scope="module")
def hf_model():
    """Load HuggingFace model as reference."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16
    ).cuda()
    return model, tokenizer

@pytest.fixture(scope="module")
def vllm_model():
    """Load vLLM model."""
    return LLM(
        model=MODEL_NAME,
        dtype="float16",
        max_model_len=512,
    )

def test_greedy_decoding(hf_model, vllm_model):
    """Test that greedy decoding matches HuggingFace."""
    hf_model_obj, tokenizer = hf_model
    
    # Get HuggingFace outputs
    hf_outputs = []
    for prompt in PROMPTS:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output = hf_model_obj.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                do_sample=False,  # Greedy
            )
        hf_text = tokenizer.decode(output[0], skip_special_tokens=True)
        hf_outputs.append(hf_text)
    
    # Get vLLM outputs
    sampling_params = SamplingParams(
        temperature=0,  # Greedy
        max_tokens=MAX_TOKENS,
    )
    vllm_outputs = vllm_model.generate(PROMPTS, sampling_params)
    vllm_texts = [
        prompt + output.outputs[0].text
        for prompt, output in zip(PROMPTS, vllm_outputs)
    ]
    
    # Compare
    for prompt, hf_text, vllm_text in zip(PROMPTS, hf_outputs, vllm_texts):
        assert hf_text == vllm_text, (
            f"Mismatch for prompt: {prompt}\\n"
            f"HF:   {hf_text}\\n"
            f"vLLM: {vllm_text}"
        )

@pytest.mark.parametrize("dtype", ["float16", "bfloat16"])
def test_output_dtypes(dtype):
    """Test model works with different dtypes."""
    llm = LLM(model=MODEL_NAME, dtype=dtype, max_model_len=256)
    params = SamplingParams(temperature=0, max_tokens=10)
    outputs = llm.generate(["Hello"], params)
    assert len(outputs) == 1
    assert len(outputs[0].outputs[0].text) > 0
'''

print("Model Correctness Test Template (from vLLM tests/models/):")
print("=" * 70)
print(model_test_template)

In [ ]:
# Demo: Simulated model correctness test (runs without GPU)

import random
import hashlib

class MockTokenizer:
    """Mock tokenizer for testing without a real model."""
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
    
    def encode(self, text):
        # Deterministic encoding based on text hash
        h = hashlib.md5(text.encode()).hexdigest()
        random.seed(int(h, 16) % (2**32))
        length = max(3, len(text.split()))
        return [random.randint(1, self.vocab_size - 1) for _ in range(length)]
    
    def decode(self, token_ids):
        words = ["the", "a", "is", "was", "and", "of", "to", "in",
                 "that", "it", "for", "with", "on", "at", "by"]
        return " ".join(words[tid % len(words)] for tid in token_ids)

class MockModel:
    """Mock model for deterministic testing."""
    def __init__(self, vocab_size=1000, seed=42):
        self.vocab_size = vocab_size
        self.seed = seed
    
    def generate(self, input_ids, max_tokens=10, temperature=0.0):
        """Generate tokens deterministically."""
        # Use input hash as seed for deterministic output
        seed = sum(input_ids) + self.seed
        random.seed(seed)
        return [random.randint(1, self.vocab_size - 1) for _ in range(max_tokens)]

class MockVLLM(MockModel):
    """Mock vLLM model (should produce same outputs as MockModel)."""
    pass

def test_greedy_equivalence():
    """Test that both 'backends' produce same output with greedy decoding."""
    tokenizer = MockTokenizer()
    hf_model = MockModel(seed=42)
    vllm_model = MockVLLM(seed=42)
    
    prompts = [
        "Hello, my name is",
        "The capital of France is",
        "Once upon a time",
    ]
    
    print("Greedy Decoding Equivalence Test")
    print("=" * 50)
    
    all_pass = True
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt)
        
        hf_output = hf_model.generate(input_ids, max_tokens=10, temperature=0)
        vllm_output = vllm_model.generate(input_ids, max_tokens=10, temperature=0)
        
        hf_text = tokenizer.decode(hf_output)
        vllm_text = tokenizer.decode(vllm_output)
        
        match = hf_output == vllm_output
        status = "PASS" if match else "FAIL"
        all_pass = all_pass and match
        
        print(f"\n[{status}] Prompt: '{prompt}'")
        print(f"  HF:   {hf_text}")
        print(f"  vLLM: {vllm_text}")
    
    print(f"\n{'=' * 50}")
    print(f"Overall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")
    return all_pass

test_greedy_equivalence()

## 4. Integration Tests: API Server Testing

In [ ]:
# Demo: E2E API test template

api_test_template = '''
"""End-to-end API server test.

Pattern used in vLLM's tests/entrypoints/test_openai_server.py.
"""
import pytest
import subprocess
import time
import requests
import json
import openai

MODEL_NAME = "facebook/opt-125m"
PORT = 8100  # Use non-standard port for testing

@pytest.fixture(scope="module")
def server():
    """Start the vLLM server as a subprocess."""
    proc = subprocess.Popen(
        [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", MODEL_NAME,
            "--port", str(PORT),
            "--max-model-len", "256",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    
    # Wait for server to be ready
    for _ in range(120):
        try:
            resp = requests.get(f"http://localhost:{PORT}/health")
            if resp.status_code == 200:
                break
        except requests.ConnectionError:
            time.sleep(1)
    else:
        proc.kill()
        raise RuntimeError("Server failed to start")
    
    yield f"http://localhost:{PORT}"
    
    # Cleanup
    proc.terminate()
    proc.wait()

def test_completions_endpoint(server):
    """Test the /v1/completions endpoint."""
    resp = requests.post(
        f"{server}/v1/completions",
        json={
            "model": MODEL_NAME,
            "prompt": "Hello, world!",
            "max_tokens": 10,
            "temperature": 0,
        }
    )
    assert resp.status_code == 200
    data = resp.json()
    assert "choices" in data
    assert len(data["choices"]) == 1
    assert len(data["choices"][0]["text"]) > 0

def test_chat_completions_endpoint(server):
    """Test the /v1/chat/completions endpoint."""
    resp = requests.post(
        f"{server}/v1/chat/completions",
        json={
            "model": MODEL_NAME,
            "messages": [
                {"role": "user", "content": "Say hello"}
            ],
            "max_tokens": 10,
        }
    )
    assert resp.status_code == 200
    data = resp.json()
    assert "choices" in data

def test_streaming(server):
    """Test streaming completions."""
    resp = requests.post(
        f"{server}/v1/completions",
        json={
            "model": MODEL_NAME,
            "prompt": "Once upon a time",
            "max_tokens": 20,
            "stream": True,
        },
        stream=True,
    )
    assert resp.status_code == 200
    chunks = []
    for line in resp.iter_lines():
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: ") and line != "data: [DONE]":
                chunks.append(json.loads(line[6:]))
    assert len(chunks) > 0

def test_models_endpoint(server):
    """Test the /v1/models endpoint."""
    resp = requests.get(f"{server}/v1/models")
    assert resp.status_code == 200
    data = resp.json()
    assert len(data["data"]) > 0
    assert data["data"][0]["id"] == MODEL_NAME
'''

print("E2E API Test Template:")
print("=" * 60)
print(api_test_template)

In [ ]:
# Demo: Simulated API test using a mock server

from http.server import HTTPServer, BaseHTTPRequestHandler
import threading
import json
import urllib.request

class MockVLLMHandler(BaseHTTPRequestHandler):
    """Mock vLLM API server for testing."""
    
    def do_GET(self):
        if self.path == "/health":
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b"OK")
        elif self.path == "/v1/models":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            response = {
                "data": [{"id": "facebook/opt-125m", "object": "model"}]
            }
            self.wfile.write(json.dumps(response).encode())
    
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(content_length))
        
        if self.path == "/v1/completions":
            response = {
                "id": "cmpl-test",
                "object": "text_completion",
                "choices": [{
                    "text": " This is a test response.",
                    "index": 0,
                    "finish_reason": "stop"
                }],
                "usage": {"prompt_tokens": 5, "completion_tokens": 6, "total_tokens": 11}
            }
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps(response).encode())
    
    def log_message(self, format, *args):
        pass  # Suppress logging

# Start mock server
port = 8765
server = HTTPServer(('localhost', port), MockVLLMHandler)
thread = threading.Thread(target=server.serve_forever)
thread.daemon = True
thread.start()

print(f"Mock vLLM server started on port {port}")
print("Running API tests...\n")

# Test 1: Health check
try:
    resp = urllib.request.urlopen(f"http://localhost:{port}/health")
    assert resp.status == 200
    print("[PASS] Health check")
except Exception as e:
    print(f"[FAIL] Health check: {e}")

# Test 2: Models endpoint
try:
    resp = urllib.request.urlopen(f"http://localhost:{port}/v1/models")
    data = json.loads(resp.read())
    assert len(data["data"]) > 0
    print(f"[PASS] Models endpoint: found {data['data'][0]['id']}")
except Exception as e:
    print(f"[FAIL] Models endpoint: {e}")

# Test 3: Completions endpoint
try:
    req = urllib.request.Request(
        f"http://localhost:{port}/v1/completions",
        data=json.dumps({
            "model": "facebook/opt-125m",
            "prompt": "Hello",
            "max_tokens": 10
        }).encode(),
        headers={"Content-Type": "application/json"}
    )
    resp = urllib.request.urlopen(req)
    data = json.loads(resp.read())
    assert "choices" in data
    assert len(data["choices"][0]["text"]) > 0
    print(f"[PASS] Completions: got '{data['choices'][0]['text']}'")
except Exception as e:
    print(f"[FAIL] Completions: {e}")

server.shutdown()
print("\nMock server stopped.")

## 5. Test Fixtures and Mocking

In [ ]:
# Demo: Fixtures and mocking patterns for GPU-dependent code

from unittest.mock import MagicMock, patch, PropertyMock
import unittest

# Example: Mocking GPU operations for CPU-only testing

class GPUMemoryManager:
    """Simplified GPU memory manager (mimics vLLM's block allocator)."""
    
    def __init__(self, num_blocks: int, block_size: int):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))
        self.allocated = {}  # request_id -> [block_ids]
    
    def allocate(self, request_id: str, num_blocks: int) -> List[int]:
        if len(self.free_blocks) < num_blocks:
            raise RuntimeError("Out of memory: not enough free blocks")
        blocks = [self.free_blocks.pop(0) for _ in range(num_blocks)]
        self.allocated[request_id] = blocks
        return blocks
    
    def free(self, request_id: str):
        if request_id in self.allocated:
            blocks = self.allocated.pop(request_id)
            self.free_blocks.extend(blocks)
            self.free_blocks.sort()
    
    @property
    def num_free_blocks(self):
        return len(self.free_blocks)
    
    def get_usage_ratio(self):
        return 1.0 - (self.num_free_blocks / self.num_blocks)


class TestGPUMemoryManager(unittest.TestCase):
    """Tests for GPU memory manager using mocks."""
    
    def setUp(self):
        # Create a manager with 10 blocks of size 16
        self.manager = GPUMemoryManager(num_blocks=10, block_size=16)
    
    def test_allocate_basic(self):
        blocks = self.manager.allocate("req-1", 3)
        self.assertEqual(len(blocks), 3)
        self.assertEqual(self.manager.num_free_blocks, 7)
    
    def test_allocate_oom(self):
        self.manager.allocate("req-1", 8)
        with self.assertRaises(RuntimeError):
            self.manager.allocate("req-2", 5)  # Only 2 left
    
    def test_free_returns_blocks(self):
        self.manager.allocate("req-1", 5)
        self.assertEqual(self.manager.num_free_blocks, 5)
        self.manager.free("req-1")
        self.assertEqual(self.manager.num_free_blocks, 10)
    
    def test_usage_ratio(self):
        self.assertAlmostEqual(self.manager.get_usage_ratio(), 0.0)
        self.manager.allocate("req-1", 5)
        self.assertAlmostEqual(self.manager.get_usage_ratio(), 0.5)
        self.manager.allocate("req-2", 5)
        self.assertAlmostEqual(self.manager.get_usage_ratio(), 1.0)
    
    def test_multiple_allocate_free(self):
        """Test allocation/deallocation pattern."""
        self.manager.allocate("req-1", 3)
        self.manager.allocate("req-2", 3)
        self.manager.allocate("req-3", 3)
        self.assertEqual(self.manager.num_free_blocks, 1)
        
        self.manager.free("req-2")  # Free middle allocation
        self.assertEqual(self.manager.num_free_blocks, 4)
        
        # Should be able to allocate 4 blocks now
        blocks = self.manager.allocate("req-4", 4)
        self.assertEqual(len(blocks), 4)

# Run the tests
print("Running GPUMemoryManager Tests (mocked, no GPU needed)")
print("=" * 60)
suite = unittest.TestLoader().loadTestsFromTestCase(TestGPUMemoryManager)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

In [ ]:
# Demo: Mocking torch.cuda for CPU-only test execution
from unittest.mock import MagicMock, patch

def function_that_uses_cuda():
    """A function that normally requires CUDA."""
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")
    device = torch.device("cuda:0")
    tensor = torch.zeros(10, device=device)
    return tensor.sum().item()

# Pattern 1: Mock torch.cuda.is_available
print("Pattern 1: Mocking torch.cuda.is_available")
print("-" * 40)

mock_cuda_code = '''
@patch('torch.cuda.is_available', return_value=True)
@patch('torch.cuda.device_count', return_value=2)
@patch('torch.cuda.get_device_name', return_value='NVIDIA A100')
def test_with_mocked_cuda(mock_name, mock_count, mock_avail):
    import torch
    assert torch.cuda.is_available()
    assert torch.cuda.device_count() == 2
    assert torch.cuda.get_device_name(0) == 'NVIDIA A100'
'''
print(mock_cuda_code)

# Pattern 2: Parametrize across configs
print("\nPattern 2: Test Parametrization")
print("-" * 40)

parametrize_code = '''
@pytest.mark.parametrize("model_name,max_tokens", [
    ("facebook/opt-125m", 32),
    ("facebook/opt-350m", 32),
    ("gpt2", 64),
])
@pytest.mark.parametrize("dtype", ["float16", "bfloat16"])
def test_model_output(model_name, max_tokens, dtype):
    """Test runs for each combination of model and dtype."""
    llm = LLM(model=model_name, dtype=dtype)
    output = llm.generate(["Hello"], SamplingParams(max_tokens=max_tokens))
    assert len(output[0].outputs[0].text) > 0
'''
print(parametrize_code)

# Pattern 3: conftest.py fixtures
print("\nPattern 3: conftest.py Fixtures")
print("-" * 40)

conftest_code = '''
# tests/conftest.py
import pytest

@pytest.fixture(scope="session")
def vllm_runner():
    """Fixture that provides a reusable vLLM runner."""
    class VLLMRunner:
        def __init__(self):
            self._llm = None
            
        def run(self, model, prompts, **kwargs):
            from vllm import LLM, SamplingParams
            llm = LLM(model=model, max_model_len=256)
            params = SamplingParams(**kwargs)
            return llm.generate(prompts, params)
    
    return VLLMRunner()

@pytest.fixture
def sample_prompts():
    """Common test prompts."""
    return [
        "Hello, my name is",
        "The capital of France is",
        "In a galaxy far far away",
    ]
'''
print(conftest_code)

## 6. Test Parametrization for Multiple Models

In [ ]:
# Demo: Advanced test parametrization patterns
import unittest
from typing import List, Tuple

# Simulated model configurations for parametrized testing
MODEL_CONFIGS = [
    {"name": "opt-125m", "vocab_size": 50272, "hidden_size": 768, "num_layers": 12},
    {"name": "gpt2", "vocab_size": 50257, "hidden_size": 768, "num_layers": 12},
    {"name": "llama-7b", "vocab_size": 32000, "hidden_size": 4096, "num_layers": 32},
]

SAMPLING_CONFIGS = [
    {"temperature": 0.0, "top_p": 1.0, "name": "greedy"},
    {"temperature": 0.8, "top_p": 0.95, "name": "sampling"},
    {"temperature": 1.0, "top_p": 0.5, "name": "top_p_sampling"},
]

def validate_model_output(model_config, sampling_config, prompt, output):
    """Validate that model output meets basic requirements."""
    errors = []
    
    # Check output is not empty
    if not output:
        errors.append("Empty output")
    
    # Check all tokens are in valid range
    for token in output:
        if token < 0 or token >= model_config["vocab_size"]:
            errors.append(f"Token {token} out of range [0, {model_config['vocab_size']})")
    
    # For greedy decoding, output should be deterministic
    if sampling_config["temperature"] == 0.0:
        # Run again and compare
        pass  # In real test, would verify determinism
    
    return errors

# Run parametrized tests
print("Parametrized Model Tests")
print("=" * 70)

import random
total_tests = 0
passed_tests = 0

for model_cfg in MODEL_CONFIGS:
    for sampling_cfg in SAMPLING_CONFIGS:
        total_tests += 1
        test_name = f"{model_cfg['name']} + {sampling_cfg['name']}"
        
        # Simulate model output
        random.seed(42 if sampling_cfg["temperature"] == 0 else None)
        fake_output = [random.randint(0, model_cfg["vocab_size"] - 1) for _ in range(10)]
        
        errors = validate_model_output(model_cfg, sampling_cfg, "Hello", fake_output)
        
        if errors:
            print(f"  [FAIL] {test_name}: {errors}")
        else:
            print(f"  [PASS] {test_name}")
            passed_tests += 1

print(f"\nResults: {passed_tests}/{total_tests} passed")

---
## Hands-On Assignments

### Assignment 1: Write Unit Tests for a Scheduler Component

**Objective:** Write comprehensive unit tests for the `PriorityScheduler` class below.

**Tasks:**
1. Test basic scheduling with priority ordering
2. Test preemption when a high-priority request arrives
3. Test edge cases (empty queue, max capacity)
4. Test the aging mechanism
5. Achieve at least 90% code coverage

In [ ]:
# Assignment 1: Starter Code - PriorityScheduler to test

from dataclasses import dataclass, field
from typing import List, Optional
import time
from enum import IntEnum

class Priority(IntEnum):
    LOW = 0
    MEDIUM = 1
    HIGH = 2
    CRITICAL = 3

@dataclass
class Request:
    request_id: str
    priority: Priority
    num_tokens: int
    arrival_time: float = field(default_factory=time.time)
    wait_time: float = 0.0

class PriorityScheduler:
    """A scheduler that considers request priority with aging."""
    
    def __init__(self, max_batch_size: int = 4, max_tokens: int = 512,
                 aging_boost_interval: float = 10.0):
        self.max_batch_size = max_batch_size
        self.max_tokens = max_tokens
        self.aging_boost_interval = aging_boost_interval
        self.waiting: List[Request] = []
        self.running: List[Request] = []
    
    def add_request(self, request: Request):
        self.waiting.append(request)
    
    def _effective_priority(self, request: Request) -> float:
        """Calculate effective priority with aging."""
        age_boost = request.wait_time / self.aging_boost_interval
        return request.priority.value + age_boost
    
    def schedule(self, current_time: Optional[float] = None) -> dict:
        """Schedule the next batch of requests."""
        if current_time is None:
            current_time = time.time()
        
        # Update wait times
        for req in self.waiting:
            req.wait_time = current_time - req.arrival_time
        
        # Sort by effective priority (highest first)
        self.waiting.sort(key=lambda r: self._effective_priority(r), reverse=True)
        
        # Select requests for the batch
        scheduled = []
        total_tokens = sum(r.num_tokens for r in self.running)
        
        remaining_waiting = []
        for req in self.waiting:
            if (len(self.running) + len(scheduled) < self.max_batch_size and
                total_tokens + req.num_tokens <= self.max_tokens):
                scheduled.append(req)
                total_tokens += req.num_tokens
            else:
                remaining_waiting.append(req)
        
        self.waiting = remaining_waiting
        self.running.extend(scheduled)
        
        return {
            "scheduled": [r.request_id for r in scheduled],
            "num_running": len(self.running),
            "num_waiting": len(self.waiting),
        }
    
    def complete_request(self, request_id: str):
        self.running = [r for r in self.running if r.request_id != request_id]


# TODO: Write your tests here
class TestPriorityScheduler(unittest.TestCase):
    
    def setUp(self):
        self.scheduler = PriorityScheduler(
            max_batch_size=3, max_tokens=200, aging_boost_interval=10.0
        )
    
    def test_basic_scheduling(self):
        """TODO: Test that requests are scheduled by priority."""
        # Add requests with different priorities
        # Schedule and verify order
        pass
    
    def test_max_batch_size(self):
        """TODO: Test that max_batch_size is respected."""
        pass
    
    def test_max_tokens_limit(self):
        """TODO: Test that max_tokens limit is respected."""
        pass
    
    def test_aging_boosts_priority(self):
        """TODO: Test that waiting requests get priority boost."""
        # A LOW priority request that has waited 30s should
        # have higher effective priority than a MEDIUM request
        # that just arrived (with aging_boost_interval=10.0)
        pass
    
    def test_empty_queue(self):
        """TODO: Test scheduling with empty queue."""
        pass
    
    def test_complete_request_frees_slot(self):
        """TODO: Test that completing a request frees a batch slot."""
        pass

# Run your tests
# suite = unittest.TestLoader().loadTestsFromTestCase(TestPriorityScheduler)
# unittest.TextTestRunner(verbosity=2).run(suite)
print("Complete the TestPriorityScheduler class and run the tests.")

### Assignment 2: Write a Model Correctness Test

**Objective:** Write a test that verifies model output consistency between
different configurations.

**Tasks:**
1. Compare greedy outputs between float16 and bfloat16
2. Compare outputs between single-request and batched inference
3. Verify output token probabilities are valid
4. Test with different sequence lengths

In [ ]:
# Assignment 2: Starter Code

import math
import random

class SimulatedLLM:
    """Simulated LLM for testing without GPU."""
    
    def __init__(self, vocab_size=100, dtype="float16", seed=42):
        self.vocab_size = vocab_size
        self.dtype = dtype
        self.seed = seed
    
    def generate(self, prompt_tokens: List[int], max_tokens: int = 10,
                 temperature: float = 0.0) -> dict:
        """Generate tokens and log-probabilities."""
        random.seed(self.seed + sum(prompt_tokens))
        
        output_tokens = []
        logprobs = []
        
        for _ in range(max_tokens):
            # Simulate logits
            raw_logits = [random.gauss(0, 1) for _ in range(self.vocab_size)]
            
            # Add dtype-dependent noise
            if self.dtype == "float16":
                noise_scale = 1e-4
            else:  # bfloat16
                noise_scale = 1e-3
            
            logits = [l + random.gauss(0, noise_scale) for l in raw_logits]
            
            # Softmax
            max_logit = max(logits)
            exp_logits = [math.exp(l - max_logit) for l in logits]
            total = sum(exp_logits)
            probs = [e / total for e in exp_logits]
            
            if temperature == 0:
                token = probs.index(max(probs))
            else:
                # Sample with temperature
                scaled = [p ** (1 / temperature) for p in probs]
                total = sum(scaled)
                scaled = [s / total for s in scaled]
                r = random.random()
                cumsum = 0
                for idx, p in enumerate(scaled):
                    cumsum += p
                    if r <= cumsum:
                        token = idx
                        break
            
            output_tokens.append(token)
            logprobs.append(math.log(probs[token]))
        
        return {
            "tokens": output_tokens,
            "logprobs": logprobs,
        }
    
    def generate_batch(self, prompt_list: List[List[int]], **kwargs) -> List[dict]:
        """Generate for a batch of prompts."""
        return [self.generate(p, **kwargs) for p in prompt_list]


class TestModelCorrectness(unittest.TestCase):
    """TODO: Write model correctness tests."""
    
    def test_greedy_determinism(self):
        """TODO: Verify greedy decoding is deterministic."""
        # Run generate twice with same input, assert same output
        pass
    
    def test_dtype_consistency(self):
        """TODO: Compare fp16 vs bf16 outputs (should be close but may differ)."""
        # Create two models with different dtypes
        # Compare outputs - greedy tokens should mostly match
        pass
    
    def test_batch_vs_single(self):
        """TODO: Verify batched inference matches single inference."""
        pass
    
    def test_logprobs_valid(self):
        """TODO: Verify log-probabilities are valid (negative, finite)."""
        pass
    
    def test_variable_length_prompts(self):
        """TODO: Test with prompts of different lengths."""
        pass

# Uncomment to run:
# suite = unittest.TestLoader().loadTestsFromTestCase(TestModelCorrectness)
# unittest.TextTestRunner(verbosity=2).run(suite)
print("Complete the TestModelCorrectness class using the SimulatedLLM.")

### Assignment 3: Write an E2E API Test

**Objective:** Write a complete E2E test suite for a mock API server.

**Tasks:**
1. Test the completions endpoint with various parameters
2. Test error handling (invalid model, bad parameters)
3. Test concurrent requests
4. Test streaming responses
5. Measure response times

In [ ]:
# Assignment 3: Starter Code

from http.server import HTTPServer, BaseHTTPRequestHandler
import threading
import json
import urllib.request
import time
from concurrent.futures import ThreadPoolExecutor

class EnhancedMockHandler(BaseHTTPRequestHandler):
    """Enhanced mock server with more realistic behavior."""
    
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(content_length)) if content_length > 0 else {}
        
        if self.path == "/v1/completions":
            # Validate required fields
            if "model" not in body:
                self._send_error(400, "model is required")
                return
            if body.get("model") not in ["test-model"]:
                self._send_error(404, f"Model '{body['model']}' not found")
                return
            if body.get("max_tokens", 10) > 1000:
                self._send_error(400, "max_tokens too large")
                return
            
            # Simulate processing time
            time.sleep(0.01)
            
            response = {
                "id": f"cmpl-{int(time.time()*1000)}",
                "choices": [{
                    "text": " generated text here",
                    "index": 0,
                    "finish_reason": "stop"
                }],
                "usage": {"prompt_tokens": 5, "completion_tokens": 4, "total_tokens": 9}
            }
            self._send_json(200, response)
        else:
            self._send_error(404, "Not found")
    
    def _send_json(self, status, data):
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(data).encode())
    
    def _send_error(self, status, message):
        self._send_json(status, {"error": {"message": message}})
    
    def log_message(self, format, *args):
        pass


class APITestSuite:
    """TODO: Complete this E2E test suite."""
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.results = []
    
    def _post(self, path, data):
        req = urllib.request.Request(
            f"{self.base_url}{path}",
            data=json.dumps(data).encode(),
            headers={"Content-Type": "application/json"}
        )
        try:
            resp = urllib.request.urlopen(req)
            return resp.status, json.loads(resp.read())
        except urllib.error.HTTPError as e:
            return e.code, json.loads(e.read())
    
    def test_valid_completion(self):
        """TODO: Test a valid completion request."""
        pass
    
    def test_invalid_model(self):
        """TODO: Test with an invalid model name."""
        pass
    
    def test_missing_model(self):
        """TODO: Test with missing model field."""
        pass
    
    def test_concurrent_requests(self):
        """TODO: Send multiple requests concurrently and verify all succeed."""
        pass
    
    def test_response_time(self):
        """TODO: Verify response time is within acceptable range."""
        pass
    
    def run_all(self):
        tests = [
            self.test_valid_completion,
            self.test_invalid_model,
            self.test_missing_model,
            self.test_concurrent_requests,
            self.test_response_time,
        ]
        for test in tests:
            name = test.__name__
            try:
                test()
                print(f"[PASS] {name}")
            except Exception as e:
                print(f"[FAIL] {name}: {e}")

# Start the mock server
port = 8766
server = HTTPServer(('localhost', port), EnhancedMockHandler)
thread = threading.Thread(target=server.serve_forever)
thread.daemon = True
thread.start()

# Run the test suite
# suite = APITestSuite(f"http://localhost:{port}")
# suite.run_all()

print(f"Mock server running on port {port}")
print("Complete the APITestSuite class and run all tests.")

server.shutdown()

---
## Summary

In this notebook, we covered:

1. **Test organization** in vLLM and SGLang projects
2. **Unit tests** for core scheduler and memory management components
3. **Model correctness tests** comparing outputs with reference implementations
4. **E2E API tests** with server fixtures and mock servers
5. **Fixtures and mocking** for GPU-dependent code
6. **Test parametrization** across multiple models and configurations

**Next:** Chapter 11.3 - Debugging Techniques